In [1]:
import pandas as pd
import json
from pathlib import Path

In [2]:
INPUT_FILES = [
    (
        "bilstm_redewiedergabe",
        Path("bilstm_evaluation/bilstm_predictions_redewiedergabe_2026_10_20-14_06_50.json"),
    ),
    (
        "bilstm_hamburger_anzeiger",
        Path("bilstm_evaluation/bilstm_predictions_hamburger_anzeiger_2026_10_20-14_16_58.json")
    ),
    (
        "roberta",
        Path("roberta_evaluation/roberta_predictions_2026_10_20-18_21_42.json")
    )
]

## Span-Errors

In [3]:
all_rows = []

for model_name, file_path in INPUT_FILES:
    with file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    for khm_key, khm_entry in sorted(
        data.items(), key=lambda kv: int(kv[0])
    ):
        khm = int(khm_entry.get("khm", khm_key))

        true_spans = []
        predicted_spans = []

        # RoBERTa: Spans liegen direkt auf Märchenebene
        if model_name.lower() == "roberta":
            for span in khm_entry.get("true_spans", []):
                true_spans.append(dict(span))

            for span in khm_entry.get("predicted_spans", []):
                predicted_spans.append(dict(span))

        # BiLSTM: Spans liegen innerhalb der einzelnen Sätze
        else:
            for sentence in khm_entry.get("sentences", []):
                for span in sentence.get("true_spans", []):
                    true_spans.append({
                        **span,
                        "sentence_id": sentence.get("sentence_id"),
                        "id": sentence.get("id"),
                    })

                for span in sentence.get("predicted_spans", []):
                    predicted_spans.append({
                        **span,
                        "sentence_id": sentence.get("sentence_id"),
                        "id": sentence.get("id"),
                    })

        all_rows.append({
            "khm": khm,
            "model": model_name,
            "true_spans": true_spans,
            "predicted_spans": predicted_spans,
        })

all_rows.sort(key=lambda row: (row["khm"], row["model"]))

out_path = Path("spans_by_khm.json")
out_path.write_text(
    json.dumps(all_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

545793

In [4]:
INPUT_FILE = Path("spans_by_khm.json")

with INPUT_FILE.open("r", encoding="utf-8") as f:
    data = json.load(f)

# Falls die Datei versehentlich nur ein einzelnes Objekt enthält
if isinstance(data, dict):
    data = [data]

df_input = pd.DataFrame(data)


# --------------------------------------------------
# 2. Span-Auswertung pro Märchen und Modell
# --------------------------------------------------

summary_rows = []
detail_rows = []

for _, entry in df_input.iterrows():
    khm = int(entry["khm"])
    model = entry["model"]

    true_spans = entry.get("true_spans", [])
    predicted_spans = entry.get("predicted_spans", [])

    # Fehlende Werte aus der DataFrame-Zelle abfangen
    if not isinstance(true_spans, list):
        true_spans = []

    if not isinstance(predicted_spans, list):
        predicted_spans = []

    is_roberta = model.lower() == "roberta"

    def span_key(span):
        if is_roberta:
            return (span["start"], span["end"])

        return (
            span["sentence_id"],
            span["start"],
            span["end"],
        )

    # Eindeutigkeit der Schlüssel prüfen
    true_key_list = [span_key(s) for s in true_spans]
    pred_key_list = [span_key(s) for s in predicted_spans]

    if len(true_key_list) != len(set(true_key_list)):
        raise ValueError(
            f"Doppelte True-Spans: khm={khm}, model={model}"
        )

    if len(pred_key_list) != len(set(pred_key_list)):
        raise ValueError(
            f"Doppelte Predicted-Spans: khm={khm}, model={model}"
        )

    true_by_key = {
        span_key(s): s for s in true_spans
    }
    pred_by_key = {
        span_key(s): s for s in predicted_spans
    }

    true_keys = set(true_by_key)
    pred_keys = set(pred_by_key)

    shared_keys = true_keys & pred_keys

    # Gleiche Grenzen, gleiches Label
    correct_keys = {
        k for k in shared_keys
        if true_by_key[k]["label"] == pred_by_key[k]["label"]
    }

    # Gleiche Grenzen, aber anderes Label
    wrong_label_keys = shared_keys - correct_keys

    # Keine Prediction mit denselben Grenzen
    missed_keys = true_keys - pred_keys

    # Keine Ground Truth mit denselben Grenzen
    extra_keys = pred_keys - true_keys

    # Label-Zählungen der übersehenen Spans
    missed_fuzzy = sum(
        true_by_key[k]["label"] == "FUZZY_LOC"
        for k in missed_keys
    )

    missed_metaphoric = sum(
        true_by_key[k]["label"] == "METAPHORIC_LOC"
        for k in missed_keys
    )

    # Label-Verwechslungen
    metaphoric_as_fuzzy = sum(
        true_by_key[k]["label"] == "METAPHORIC_LOC"
        and pred_by_key[k]["label"] == "FUZZY_LOC"
        for k in wrong_label_keys
    )

    fuzzy_as_metaphoric = sum(
        true_by_key[k]["label"] == "FUZZY_LOC"
        and pred_by_key[k]["label"] == "METAPHORIC_LOC"
        for k in wrong_label_keys
    )

    # Label-Zählungen der Extra-Spans
    extra_metaphoric = sum(
        pred_by_key[k]["label"] == "METAPHORIC_LOC"
        for k in extra_keys
    )

    extra_fuzzy = sum(
        pred_by_key[k]["label"] == "FUZZY_LOC"
        for k in extra_keys
    )

    summary_rows.append({
        "khm": khm,
        "model": model,
        "n_true_spans": len(true_spans),
        "n_predicted_spans": len(predicted_spans),
        "correct_spans": len(correct_keys),
        "missed_spans": len(missed_keys),
        "extra_spans": len(extra_keys),
        "wrong_label_spans": len(wrong_label_keys),

        "missed_fuzzy": missed_fuzzy,
        "missed_metaphoric": missed_metaphoric,

        "metaphoric_as_fuzzy": metaphoric_as_fuzzy,
        "fuzzy_as_metaphoric": fuzzy_as_metaphoric,

        "extra_metaphoric": extra_metaphoric,
        "extra_fuzzy": extra_fuzzy,
    })

    # Fehlerdetails speichern
    for k in missed_keys:
        span = true_by_key[k]
        detail_rows.append({
            "khm": khm,
            "model": model,
            "error_type": "missed",
            "true_label": span["label"],
            "predicted_label": "O",
            "text_true": span.get("text"),
            "text_predicted": None,
            "start": span["start"],
            "end": span["end"],
        })

    for k in extra_keys:
        span = pred_by_key[k]
        detail_rows.append({
            "khm": khm,
            "model": model,
            "error_type": "extra",
            "true_label": "O",
            "predicted_label": span["label"],
            "text_true": None,
            "text_predicted": span.get("text"),
            "start": span["start"],
            "end": span["end"],
        })

    for k in wrong_label_keys:
        true_span = true_by_key[k]
        pred_span = pred_by_key[k]

        detail_rows.append({
            "khm": khm,
            "model": model,
            "error_type": "wrong_label",
            "true_label": true_span["label"],
            "predicted_label": pred_span["label"],
            "text_true": true_span.get("text"),
            "text_predicted": pred_span.get("text"),
            "start": true_span["start"],
            "end": true_span["end"],
        })


# --------------------------------------------------
# 3. DataFrames erstellen und pro Märchen speichern
# --------------------------------------------------

df_summary = pd.DataFrame(summary_rows)

df_summary = df_summary.sort_values(
    ["khm", "model"]
).reset_index(drop=True)

df_details = pd.DataFrame(detail_rows)

df_summary.to_csv(
    "span_analysis_by_khm.csv",
    index=False,
    encoding="utf-8-sig",
)

df_details.to_csv(
    "span_errors_detail.csv",
    index=False,
    encoding="utf-8-sig",
)


# --------------------------------------------------
# 4. Gesamtübersicht pro Modell
# --------------------------------------------------

count_columns = [
    "n_true_spans",
    "n_predicted_spans",
    "correct_spans",
    "missed_spans",
    "extra_spans",
    "wrong_label_spans",
    "missed_fuzzy",
    "missed_metaphoric",
    "metaphoric_as_fuzzy",
    "fuzzy_as_metaphoric",
    "extra_metaphoric",
    "extra_fuzzy",
]

df_total = (
    df_summary
    .groupby("model", as_index=False)[count_columns]
    .sum()
)

df_total.to_csv(
    "span_analysis_total.csv",
    index=False,
    encoding="utf-8-sig",
)


In [5]:
df_summary.head()

,khm,model,n_true_spans,n_predicted_spans,correct_spans,missed_spans,extra_spans,wrong_label_spans,missed_fuzzy,missed_metaphoric,metaphoric_as_fuzzy,fuzzy_as_metaphoric,extra_metaphoric,extra_fuzzy
0,4,bilstm_hamburger_anzeiger,80,73,67,13,6,0,11,2,0,0,0,6
1,4,bilstm_redewiedergabe,80,73,67,13,6,0,11,2,0,0,0,6
2,4,roberta,80,78,71,9,7,0,7,2,0,0,0,7
3,5,bilstm_hamburger_anzeiger,38,32,25,10,4,3,8,2,2,1,0,4
4,5,bilstm_redewiedergabe,38,25,24,13,0,1,9,4,0,1,0,0


In [6]:
df_total.head()

,model,n_true_spans,n_predicted_spans,correct_spans,missed_spans,extra_spans,wrong_label_spans,missed_fuzzy,missed_metaphoric,metaphoric_as_fuzzy,fuzzy_as_metaphoric,extra_metaphoric,extra_fuzzy
0,bilstm_hamburger_anzeiger,593,631,486,80,118,27,67,13,14,13,9,109
1,bilstm_redewiedergabe,593,532,431,131,70,31,113,18,14,17,9,61
2,roberta,593,615,504,62,84,27,52,10,11,16,17,67


## Token Errors

In [7]:
all_rows = []

for model_name, file_path in INPUT_FILES:
    with file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    for khm_key, khm_entry in sorted(
        data.items(), key=lambda kv: int(kv[0])
    ):
        khm = int(khm_entry.get("khm", khm_key))
        token_errors = []

        # ------------------------------------------
        # Transformer: fehlerhafte Tokens herausfiltern
        # ------------------------------------------
        if model_name.lower() == "roberta":
            for token_index, token in enumerate(
                khm_entry.get("tokens", [])
            ):
                true_tag = token.get("true_tag")
                predicted_tag = token.get("predicted_tag")

                # Nur fehlerhafte Tokens übernehmen
                if true_tag != predicted_tag:
                    token_errors.append({
                        "token_index": token_index,
                        "token": token.get("text"),
                        "true_tag": true_tag,
                        "predicted_tag": predicted_tag,
                        "start": token.get("start"),
                        "end": token.get("end"),
                        "confidence": token.get("confidence"),
                    })
        else:
            # ------------------------------------------
            # BiLSTM: token_errors aus den Sätzen lesen
            # ------------------------------------------
            if "sentences" in khm_entry:
                for sentence in khm_entry.get("sentences", []):
                    for err in sentence.get("token_errors", []):
                        token_errors.append({
                            "sentence_id": sentence.get("sentence_id"),
                            "id": sentence.get("id"),
                            "token_index": err.get("token_index"),
                            "token": err.get("token"),
                            "true_tag": err.get("true_tag"),
                            "predicted_tag": err.get("predicted_tag"),
                        })


        all_rows.append({
            "khm": khm,
            "model": model_name,
            "token_errors": token_errors,
        })


# Nach Märchen und Modell sortieren
all_rows.sort(key=lambda row: (row["khm"], row["model"]))

# Zusammenfassungs-JSON speichern
out_path = Path("token_errors_by_khm.json")
out_path.write_text(
    json.dumps(all_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Anzahl Märchen-Modell-Kombinationen: {len(all_rows)}")
print(
    "Anzahl Token-Fehler insgesamt:",
    sum(len(row["token_errors"]) for row in all_rows)
)

Anzahl Märchen-Modell-Kombinationen: 54
Anzahl Token-Fehler insgesamt: 1770


In [8]:
ENTITY_LABELS = {
    "B-FUZZY_LOC",
    "I-FUZZY_LOC",
    "B-METAPHORIC_LOC",
    "I-METAPHORIC_LOC",
}


def get_entity_type(tag):
    """Gibt FUZZY_LOC oder METAPHORIC_LOC zurück, sonst None."""
    if tag in ENTITY_LABELS:
        return tag[2:]  # Entfernt B- oder I-
    return None


def analyze_token(true_tag, predicted_tag):
    """Ordnet einen Token-Vergleich einer Fehlerkategorie zu."""
    true_type = get_entity_type(true_tag)
    predicted_type = get_entity_type(predicted_tag)

    if true_tag == predicted_tag:
        return "correct"

    # Entity im Goldstandard wurde als O vorhergesagt.
    if true_type is not None and predicted_tag == "O":
        return "missed"

    # Vorhergesagte Entity steht im Goldstandard als O.
    if true_tag == "O" and predicted_type is not None:
        return "extra"

    # Beide Tags sind Entity-Labels, stimmen aber nicht überein.
    if true_type is not None and predicted_type is not None:
        return "wrong_label"

    # Fallback für unerwartete oder fehlende Tags.
    return "other"


summary_rows = []
detail_rows = []


for model_name, file_path in INPUT_FILES:
    with Path(file_path).open("r", encoding="utf-8") as f:
        data = json.load(f)

    for khm_key, khm_entry in sorted(
        data.items(), key=lambda item: int(item[0])
    ):
        khm = int(khm_entry.get("khm", khm_key))

        counts = {
            "total_tokens": 0,
            "true_entity_tokens": 0,
            "true_fuzzy_tokens": 0,
            "true_metaphoric_tokens": 0,
            "predicted_entity_tokens": 0,
            "predicted_fuzzy_tokens": 0,
            "predicted_metaphoric_tokens": 0,
            "correct_entity_tokens": 0,
            "correct": 0,
            "missed": 0,
            "missed_fuzzy": 0,
            "missed_metaphoric": 0,
            "extra": 0,
            "extra_fuzzy": 0,
            "extra_metaphoric": 0,
            "wrong_label": 0,
            "metaphoric_as_fuzzy": 0,
            "fuzzy_as_metaphoric": 0,
            "wrong_bio_prefix_same_type": 0,
            "other": 0,
        }

        token_records = []

        # BiLSTM: Daten liegen satzweise vor.
        if "sentences" in khm_entry:
            for sentence in khm_entry.get("sentences", []):
                tokens = sentence.get("tokens", [])
                true_tags = sentence.get("true_tags", [])
                predicted_tags = sentence.get("predicted_tags", [])

                if not (
                    len(tokens) == len(true_tags) == len(predicted_tags)
                ):
                    raise ValueError(
                        f"Unterschiedliche Längen in Modell {model_name}, "
                        f"KHM {khm}, Satz {sentence.get('sentence_id')}"
                    )

                for i, (token, true_tag, predicted_tag) in enumerate(
                    zip(tokens, true_tags, predicted_tags)
                ):
                    token_records.append({
                        "sentence_id": sentence.get("sentence_id"),
                        "id": sentence.get("id"),
                        "token_index": i,
                        "token": token,
                        "true_tag": true_tag,
                        "predicted_tag": predicted_tag,
                    })

        # Transformer: Tokens liegen direkt auf Märchenebene vor.
        else:
            for i, token_data in enumerate(khm_entry.get("tokens", [])):
                token_records.append({
                    "sentence_id": None,
                    "id": None,
                    "token_index": i,
                    "token": token_data.get("text"),
                    "true_tag": token_data.get("true_tag"),
                    "predicted_tag": token_data.get("predicted_tag"),
                    "start": token_data.get("start"),
                    "end": token_data.get("end"),
                    "confidence": token_data.get("confidence"),
                })

        # Alle Tokens klassifizieren und Fehlerdetails sammeln.
        for record in token_records:
            true_tag = record["true_tag"]
            predicted_tag = record["predicted_tag"]

            true_type = get_entity_type(true_tag)
            predicted_type = get_entity_type(predicted_tag)

            # Gold-Entity-Tokens pro Typ zählen
            if true_type == "FUZZY_LOC":
                counts["true_fuzzy_tokens"] += 1
            elif true_type == "METAPHORIC_LOC":
                counts["true_metaphoric_tokens"] += 1

            # Vom Modell vorhergesagte Entity-Tokens pro Typ zählen
            if predicted_type == "FUZZY_LOC":
                counts["predicted_fuzzy_tokens"] += 1
            elif predicted_type == "METAPHORIC_LOC":
                counts["predicted_metaphoric_tokens"] += 1

            category = analyze_token(true_tag, predicted_tag)
            counts["total_tokens"] += 1

            if category == "correct":
                counts["correct"] += 1

            elif category == "missed":
                counts["missed"] += 1

                if true_type == "FUZZY_LOC":
                    counts["missed_fuzzy"] += 1
                elif true_type == "METAPHORIC_LOC":
                    counts["missed_metaphoric"] += 1

            elif category == "extra":
                counts["extra"] += 1

                if predicted_type == "FUZZY_LOC":
                    counts["extra_fuzzy"] += 1
                elif predicted_type == "METAPHORIC_LOC":
                    counts["extra_metaphoric"] += 1

            elif category == "wrong_label":
                counts["wrong_label"] += 1

                if (
                    true_type == "METAPHORIC_LOC"
                    and predicted_type == "FUZZY_LOC"
                ):
                    counts["metaphoric_as_fuzzy"] += 1

                elif (
                    true_type == "FUZZY_LOC"
                    and predicted_type == "METAPHORIC_LOC"
                ):
                    counts["fuzzy_as_metaphoric"] += 1

                elif true_type == predicted_type:
                    # Beispielsweise B-FUZZY_LOC statt I-FUZZY_LOC.
                    counts["wrong_bio_prefix_same_type"] += 1

            else:
                counts["other"] += 1

            # Alle vorhergesagten und tatsächlichen Entity-Tokens zählen.
            if predicted_type is not None:
                counts["predicted_entity_tokens"] += 1

            if true_type is not None:
                counts["true_entity_tokens"] += 1

            if true_tag == predicted_tag and true_type is not None:
                counts["correct_entity_tokens"] += 1

            # Nur übersehene, zusätzliche und falsch gelabelte Tokens
            # werden in der Detaildatei gespeichert.
            if category in {"missed", "extra", "wrong_label"}:
                detail_rows.append({
                    "khm": khm,
                    "model": model_name,
                    **record,
                    "error_type": category,
                    "true_entity_type": true_type,
                    "predicted_entity_type": predicted_type,
                })

        summary_rows.append({
            "khm": khm,
            "model": model_name,
            **counts,
        })



# ---------------------------------------------------------
# 1. Prozentwerte pro Märchen und Modell
# ---------------------------------------------------------
by_khm = pd.DataFrame(summary_rows)

count_columns = [
    col for col in by_khm.columns
    if col not in {"khm", "model"}
]

# Alle Zählspalten außer total_tokens erhalten Prozent-Spalten.
metric_columns = [
    col for col in count_columns
    if col != "total_tokens"
]

for col in metric_columns:
    by_khm[f"{col}_pct"] = (
        by_khm[col]
        .div(by_khm["total_tokens"].replace(0, pd.NA))
        .mul(100)
        .fillna(0)
        .round(2)
    )

by_khm["correct_entity_pct_of_predicted"] = (
    100 * by_khm["correct_entity_tokens"]
    / by_khm["predicted_entity_tokens"].replace(0, pd.NA)
).fillna(0).round(2)



by_khm = by_khm.sort_values(["khm", "model"])

by_khm.to_csv(
    "token_analysis_by_khm.csv",
    index=False,
    encoding="utf-8-sig",
)


# ---------------------------------------------------------
# 2. Gesamtzahlen und Prozentwerte pro Modell
# ---------------------------------------------------------
# Zuerst die absoluten Zahlen über alle Märchen summieren.
total = (
    by_khm.groupby("model", as_index=False)[count_columns]
    .sum()
    .sort_values("model")
)

# Prozente auf Basis der aufsummierten Tokenzahlen berechnen.
for col in metric_columns:
    total[f"{col}_pct"] = (
        total[col]
        .div(total["total_tokens"].replace(0, pd.NA))
        .mul(100)
        .fillna(0)
        .round(2)
    )

total["correct_entity_pct_of_predicted"] = (
    100 * total["correct_entity_tokens"]
    / total["predicted_entity_tokens"].replace(0, pd.NA)
).fillna(0).round(2)

total.to_csv(
    "token_analysis_total.csv",
    index=False,
    encoding="utf-8-sig",
)




# ---------------------------------------------------------
# 3. Einzelne Token-Fehler mit Prozentangaben
# ---------------------------------------------------------
detail_columns = [
    "khm",
    "model",
    "sentence_id",
    "id",
    "token_index",
    "token",
    "true_tag",
    "predicted_tag",
    "error_type",
    "true_entity_type",
    "predicted_entity_type",
    "start",
    "end",
    "confidence",
]

details = pd.DataFrame(detail_rows, columns=detail_columns)

# Gesamtzahl der Tokens pro Märchen und Modell hinzufügen.
denominators = by_khm[["khm", "model", "total_tokens"]].copy()

details = details.merge(
    denominators,
    on=["khm", "model"],
    how="left",
)

# Prozentanteil eines einzelnen Fehler-Tokens am gesamten Tokenbestand.
details["error_share_pct"] = (
    100 / details["total_tokens"].replace(0, pd.NA)
).fillna(0).round(4)

details = details.sort_values(
    ["khm", "model", "sentence_id", "token_index"],
    na_position="last",
)

details.to_csv(
    "token_errors_detail.csv",
    index=False,
    encoding="utf-8-sig",
)


In [9]:
# CSV einlesen
df = pd.read_csv("token_analysis_total.csv")

for _, row in df.iterrows():
    model = row["model"]

    # Bezogen auf alle Gold-Entity-Tokens
    missed_rate = (
        row["missed"] / row["true_entity_tokens"] * 100
        if row["true_entity_tokens"] > 0 else 0
    )

    correct_rate = (
        row["correct_entity_tokens"] / row["true_entity_tokens"] * 100
        if row["true_entity_tokens"] > 0 else 0
    )

    # Bezogen auf alle vorhergesagten Entity-Tokens
    extra_rate = (
        row["extra"] / row["predicted_entity_tokens"] * 100
        if row["predicted_entity_tokens"] > 0 else 0
    )

    wrong_label_rate = (
        row["wrong_label"] / row["predicted_entity_tokens"] * 100
        if row["predicted_entity_tokens"] > 0 else 0
    )

    print(f"Modell: {model}")
    print(f"  Missed-Rate:      {missed_rate:.2f}%")
    print(f"  Correct-Rate:     {correct_rate:.2f}%")
    print(f"  Extra-Rate:       {extra_rate:.2f}%")
    print(f"  Wrong-Label-Rate: {wrong_label_rate:.2f}%")
    print()

Modell: bilstm_hamburger_anzeiger
  Missed-Rate:      12.38%
  Correct-Rate:     83.12%
  Extra-Rate:       18.15%
  Wrong-Label-Rate: 4.21%

Modell: bilstm_redewiedergabe
  Missed-Rate:      20.68%
  Correct-Rate:     74.41%
  Extra-Rate:       11.56%
  Wrong-Label-Rate: 5.48%

Modell: roberta
  Missed-Rate:      9.47%
  Correct-Rate:     86.68%
  Extra-Rate:       11.06%
  Wrong-Label-Rate: 3.78%



In [10]:
df = pd.read_csv("token_analysis_total.csv")

for _, row in df.iterrows():
    print(f"\nModell: {row['model']}")

    # Missed-Rate pro Entity-Typ
    fuzzy_missed_rate = (
        row["missed_fuzzy"] / row["true_fuzzy_tokens"] * 100
        if row["true_fuzzy_tokens"] > 0 else 0
    )

    metaphoric_missed_rate = (
        row["missed_metaphoric"] / row["true_metaphoric_tokens"] * 100
        if row["true_metaphoric_tokens"] > 0 else 0
    )

    print(f"  Fuzzy Missed-Rate:      {fuzzy_missed_rate:.2f}%")
    print(f"  Metaphoric Missed-Rate: {metaphoric_missed_rate:.2f}%")



Modell: bilstm_hamburger_anzeiger
  Fuzzy Missed-Rate:      11.64%
  Metaphoric Missed-Rate: 18.13%

Modell: bilstm_redewiedergabe
  Fuzzy Missed-Rate:      20.20%
  Metaphoric Missed-Rate: 24.35%

Modell: roberta
  Fuzzy Missed-Rate:      9.01%
  Metaphoric Missed-Rate: 13.54%


In [11]:
df = pd.read_csv("token_analysis_total.csv")

for _, row in df.iterrows():
    fuzzy_extra_rate = (
        100 * row["extra_fuzzy"] / row["predicted_fuzzy_tokens"]
        if row["predicted_fuzzy_tokens"] > 0 else 0
    )

    metaphoric_extra_rate = (
        100 * row["extra_metaphoric"] / row["predicted_metaphoric_tokens"]
        if row["predicted_metaphoric_tokens"] > 0 else 0
    )

    print(f"\nModell: {row['model']}")
    print(f"  Fuzzy Extra-Rate:      {fuzzy_extra_rate:.2f}%")
    print(f"  Metaphoric Extra-Rate: {metaphoric_extra_rate:.2f}%")


Modell: bilstm_hamburger_anzeiger
  Fuzzy Extra-Rate:      18.73%
  Metaphoric Extra-Rate: 12.72%

Modell: bilstm_redewiedergabe
  Fuzzy Extra-Rate:      11.35%
  Metaphoric Extra-Rate: 13.25%

Modell: roberta
  Fuzzy Extra-Rate:      10.22%
  Metaphoric Extra-Rate: 18.18%


## Beispiel: KHM 115

In [12]:
base = Path("/home/pia/PycharmProjects/spatial_extraction_fairytales/evaluation")
source = base / "spans_by_khm.json"
target = base / "khm_115_correct_span_predictions.csv"

with source.open("r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for entry in data:
    if int(entry.get("khm", -1)) != 115:
        continue

    model = entry["model"]
    true_spans = entry.get("true_spans", []) or []
    pred_spans = entry.get("predicted_spans", []) or []

    is_roberta = model.lower() == "roberta"

    def span_key(span):
        if is_roberta:
            return (span["start"], span["end"])
        return (
            span.get("sentence_id"),
            span["start"],
            span["end"],
        )

    true_by_key = {span_key(span): span for span in true_spans}
    pred_by_key = {span_key(span): span for span in pred_spans}

    for key, true_span in true_by_key.items():
        pred_span = pred_by_key.get(key)
        if pred_span is None:
            continue

        if true_span.get("label") != pred_span.get("label"):
            continue

        rows.append({
            "model": model,
            "true_label": true_span["label"],
            "predicted_label": pred_span["label"],
            "text_true": true_span.get("text"),
            "start": true_span["start"],
            "end": true_span["end"],
        })

df = pd.DataFrame(rows).sort_values(
    ["model", "start", "end"],
    kind="stable",
).reset_index(drop=True)

df.to_csv(target, index=False, encoding="utf-8-sig")

print(df.to_string(index=False))
print(f"\nSaved: {target}")

                    model     true_label predicted_label          text_true  start  end
bilstm_hamburger_anzeiger METAPHORIC_LOC  METAPHORIC_LOC        in der Welt      3    6
bilstm_hamburger_anzeiger      FUZZY_LOC       FUZZY_LOC      in die Tasche      4    7
bilstm_hamburger_anzeiger      FUZZY_LOC       FUZZY_LOC      auf dem Tisch      6    9
bilstm_hamburger_anzeiger      FUZZY_LOC       FUZZY_LOC hinter einen Busch      8   11
bilstm_hamburger_anzeiger      FUZZY_LOC       FUZZY_LOC    vor dem Fenster      9   12
bilstm_hamburger_anzeiger      FUZZY_LOC       FUZZY_LOC      in eine Stadt     10   13
bilstm_hamburger_anzeiger METAPHORIC_LOC  METAPHORIC_LOC    die ganze Stadt     37   40
bilstm_hamburger_anzeiger      FUZZY_LOC       FUZZY_LOC        an der Wand     44   47
bilstm_hamburger_anzeiger      FUZZY_LOC       FUZZY_LOC    vor das Gericht     45   48
    bilstm_redewiedergabe METAPHORIC_LOC  METAPHORIC_LOC        in der Welt      3    6
    bilstm_redewiedergabe      F

## few shot prompting

In [13]:
INPUT_FILES_FEWSHOT = [
        (
        "gemma3:27b",
        Path("ollama_evaluation/ollama_annotations/annotated_fairytales_2026_10_13-15_19_36_gemma27b.json"),
    ),
    (
        "mistral-small3.2:24b",
        Path("ollama_evaluation/ollama_annotations/annotated_fairytales_2026_10_13-15_48_59_mistral24b.json")
    ),
]

In [14]:
for model_name, file_path in INPUT_FILES_FEWSHOT:
    with file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    counts = {
        "VAGUE_LOC": 0,
        "CONCEPTUAL_LOC": 0,
    }

    for entry in data:
        if not isinstance(entry, dict):
            continue

        for span in entry.get("spans", []):
            label = span.get("label")
            if label in counts:
                counts[label] += 1

    total = sum(counts.values())
    print(f"{model_name} | {file_path.name}")
    print(f"  VAGUE_LOC: {counts['VAGUE_LOC']}")
    print(f"  CONCEPTUAL_LOC: {counts['CONCEPTUAL_LOC']}")
    print(f"  total spans: {total}")
    print()

gemma3:27b | annotated_fairytales_2026_10_13-15_19_36_gemma27b.json
  VAGUE_LOC: 647
  CONCEPTUAL_LOC: 48
  total spans: 695

mistral-small3.2:24b | annotated_fairytales_2026_10_13-15_48_59_mistral24b.json
  VAGUE_LOC: 525
  CONCEPTUAL_LOC: 60
  total spans: 585

